# Ch 20 · The Primal Solvers: CG, Newton and the Exact Linesearch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/20-solvers-cg-newton.ipynb)

Companion notebook for the chapter page `chapters/20-solvers-cg-newton.html` ([page](../chapters/20-solvers-cg-newton.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/20-solvers-cg-newton/`). Budgets switch on `ON_GPU`: the same cell runs small on a CPU and big on a Colab GPU. CPU timings say nothing about GPU performance: read your own numbers.

## 1-3. The problem, the solve flow and the gradient
The solver minimizes `f(a) = 0.5 (a - a0)' M (a - a0) + sum_i s_i(J_i a - aref_i)` over `a = qacc`. `_solve`: `_solve_init_dof` (warmstart or qacc_smooth), `init_context` (inertia_scale, Jaref, Ma, states), the first gradient `grad = Ma - qfrc_smooth - qfrc_constraint`, CG's first search, then `wp.capture_while(nsolving, _solver_iteration)`.

## 4. One iteration on a 2-dof toy
A NumPy teaching solver with MJWarp's two recipes (Newton with an exact linesearch; CG with Polak-Ribiere beta >= 0). No MJWarp call.

In [ ]:
# Example 20.5b - ex5b_toy_2dof.py
"""Example 20.5b - NumPy teaching solver on a 2-dof toy (drives the solver-iteration stepper).

The Ch 19 cost on a problem small enough to draw: 2 dofs (a = qacc), a 2x2 mass matrix M, the
unconstrained acceleration a0 = qacc_smooth and three pyramidal-style rows (active when
J_i a - aref_i < 0). Both solvers follow MJWarp's 4.0.1 recipe:
  Newton: H = M + J' diag(D active) J, search = -H^-1 grad, exact linesearch
  CG:     Mgrad = M^-1 grad, beta = max(0, grad'(Mgrad - Mgrad_prev) / (grad_prev' Mgrad_prev)),
          search = -Mgrad + beta * search, exact linesearch       (Polak-Ribiere, clamped >= 0)
and stop with the _solve_done tests (scaled by 1/trace(M)). No MJWarp call: pure NumPy.
Run: python ex5b_toy_2dof.py [--dump-json PATH]
"""
import argparse
import json

import numpy as np

M = np.array([[3.0, 1.2], [1.2, 1.0]])
A0 = np.array([-1.0, -3.0])                  # gravity-like pull: where a would go unconstrained
J = np.array([[0.0, 1.0], [0.8, 0.6], [-1.0, 0.25]])
AREF = np.array([0.5, 0.2, 0.6])
D = np.array([4.0, 6.0, 3.0])
TOL = 1e-6


def cost_grad(a):
    jaref = J @ a - AREF
    act = jaref < 0
    da = a - A0
    cost = 0.5 * da @ M @ da + 0.5 * np.sum(D[act] * jaref[act] ** 2)
    grad = M @ da + J.T @ (D * act * jaref)
    return cost, grad, act


def linesearch(a, s):
    """Exact minimizer of the convex piecewise quadratic t -> cost(a + t s)."""
    jaref, jv = J @ a - AREF, J @ s
    with np.errstate(divide="ignore", invalid="ignore"):
        k = np.where(jv != 0, -jaref / jv, np.inf)
    edges = np.concatenate([[0.0], np.sort(k[(k > 0) & np.isfinite(k)]), [np.inf]])
    for lo, hi in zip(edges[:-1], edges[1:]):
        tm = lo + 1.0 if np.isinf(hi) else 0.5 * (lo + hi)
        act = jaref + tm * jv < 0
        g = s @ M @ (a - A0) + np.sum(D[act] * jv[act] * jaref[act])
        h = s @ M @ s + np.sum(D[act] * jv[act] ** 2)
        t = -g / h
        if lo <= t <= hi:
            return t
    return 0.0


def solve(kind, maxiter=20):
    scale = 1.0 / np.trace(M)
    a = A0.copy()
    c, g, act = cost_grad(a)
    mg = np.linalg.solve(M, g)
    s = -mg if kind == "cg" else None
    frames = [{"a": a.tolist(), "cost": c, "alpha": 0.0, "grad": scale * np.linalg.norm(g),
               "active": act.astype(int).tolist(), "line": 1}]
    for it in range(1, maxiter + 1):
        if kind == "newton":
            H = M + J.T @ (D[:, None] * act[:, None] * J)
            s = -np.linalg.solve(H, g)
        t = linesearch(a, s)
        a = a + t * s
        c_new, g_new, act = cost_grad(a)
        improvement = scale * (c - c_new)
        if kind == "cg":
            mg_new = np.linalg.solve(M, g_new)
            beta = max(0.0, g_new @ (mg_new - mg) / max(1e-15, g @ mg))
            s = -mg_new + beta * s
            mg = mg_new
        c, g = c_new, g_new
        gn = scale * np.linalg.norm(g)
        frames.append({"a": a.tolist(), "cost": c, "alpha": t, "grad": gn,
                       "active": act.astype(int).tolist(), "line": 3})
        done = t == 0 or 0 < improvement < TOL or gn < TOL
        if kind == "newton":
            H = M + J.T @ (D[:, None] * act[:, None] * J)
            done = done or 0.5 * scale * g @ np.linalg.solve(H, g) < TOL
        if done:
            break
    return frames


out = {"M": M.tolist(), "a0": A0.tolist(), "J": J.tolist(), "aref": AREF.tolist(),
       "D": D.tolist(), "runs": {}}
for kind in ("newton", "cg"):
    fr = solve(kind)
    out["runs"][kind] = [{k: (np.round(v, 5).tolist() if isinstance(v, (list, np.ndarray))
                              else (round(float(v), 6) if isinstance(v, float) else v))
                          for k, v in f.items()} for f in fr]
    print(f"{kind}: {len(fr) - 1} iterations")
    with np.printoptions(precision=4, suppress=True):   # local: no global print state
        for i, f in enumerate(fr):
            print(f"  {i}: a = {np.array(f['a'])}  cost {f['cost']:9.4f}  alpha {f['alpha']:.4f}  "
                  f"|grad|*scale {f['grad']:.2e}  active rows {f['active']}")

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()
if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

The same Newton in float64 on the problem MJWarp built for one world of a kicked `box_stack(5)`. Costs agree to 1e-8; qacc can differ by ~1e-3 because the cost is flat in some directions.

In [ ]:
# Example 20.5 - ex5_numpy_newton.py
"""Example 20.5 - a reference primal Newton solver in NumPy, checked against MJWarp's qacc.

We take the problem MJWarp builds for one world (M from mj_fullM, efc.J, efc.D, efc.aref and
qacc_smooth: the same arrays Ch 19 Example 19.3 used to rebuild the cost) and minimize
  f(a) = 0.5 (a - a0)' M (a - a0) + sum_i 0.5 D_i min(0, J_i a - aref_i)^2     (pyramidal rows)
with Newton's method: H = M + J' diag(D * active) J, search = -H^-1 grad, then an EXACT 1-D
linesearch that walks the kinks of the piecewise quadratic f(a + alpha * search) in order.
Both solvers start from qacc_smooth (warmstart off), so their iterates can be compared one by one.
Run: python ex5_numpy_newton.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def cost_grad(a, P):
    jaref = P["J"] @ a - P["aref"]
    act = jaref < 0.0
    da = a - P["a0"]
    cost = 0.5 * da @ P["M"] @ da + 0.5 * np.sum(P["D"][act] * jaref[act] ** 2)
    grad = P["M"] @ da + P["J"].T @ (P["D"] * act * jaref)
    return cost, grad, act


def exact_linesearch(a, s, P):
    """Minimize the convex piecewise quadratic phi(t) = f(a + t s) exactly, kink by kink."""
    jaref, jv = P["J"] @ a - P["aref"], P["J"] @ s
    g0 = s @ P["M"] @ (a - P["a0"])          # Gauss part of phi'(t) = g0 + t * h0
    h0 = s @ P["M"] @ s
    with np.errstate(divide="ignore", invalid="ignore"):
        kinks = np.where(jv != 0.0, -jaref / jv, np.inf)
    pts = np.sort(kinks[kinks > 0.0])
    lo = 0.0
    for hi in np.append(pts, np.inf):        # on (lo, hi) the active set is fixed
        t_mid = lo + 1.0 if np.isinf(hi) else 0.5 * (lo + hi)
        act = (jaref + t_mid * jv) < 0.0
        g = g0 + np.sum(P["D"][act] * jv[act] * jaref[act])   # phi'(t) = g + t * h
        h = h0 + np.sum(P["D"][act] * jv[act] ** 2)
        t = -g / h
        if lo <= t <= hi:
            return t, len(pts)
        lo = hi
    return lo, len(pts)


def numpy_newton(P, tol=1e-6, maxiter=50):
    a = P["a0"].copy()
    scale = 1.0 / np.trace(P["M"])
    history = [cost_grad(a, P)[0]]
    for it in range(1, maxiter + 1):
        c, g, act = cost_grad(a, P)
        H = P["M"] + P["J"].T @ (P["D"][:, None] * act[:, None] * P["J"])
        s = -np.linalg.solve(H, g)
        t, nk = exact_linesearch(a, s, P)
        a = a + t * s
        c_new, g_new, _ = cost_grad(a, P)
        history.append(c_new)
        print(f"  iter {it}: alpha {t:.4f}  kinks on ray {nk:2d}  cost {c_new:12.4f}  "
              f"|grad|*scale {scale * np.linalg.norm(g_new):.2e}  active {int(act.sum())}")
        if scale * (c - c_new) < tol or scale * np.linalg.norm(g_new) < tol:
            break
    return a, history


mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)
m.opt.disableflags |= mjw.DisableBit.WARMSTART      # start from qacc_smooth like the reference
d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
d.qvel.assign(np.random.default_rng(1).normal(0.0, 0.1, size=(1, mjm.nv)).astype(np.float32))
for _ in range(20):
    mjw.step(m, d)
mjw.forward(m, d)

mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d)
M = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, M)
n = int(d.nefc.numpy()[0])
P = {"M": M,
     "J": d.efc.J.numpy()[0, :n, :mjm.nv].astype(np.float64),
     "D": d.efc.D.numpy()[0, :n].astype(np.float64),
     "aref": d.efc.aref.numpy()[0, :n].astype(np.float64),
     "a0": d.qacc_smooth.numpy()[0].astype(np.float64)}
print(f"problem: nv={mjm.nv}, nefc={n} pyramidal rows, MJWarp Newton niter "
      f"{int(d.solver_niter.numpy()[0])}")
print("NumPy Newton (float64):")
a_np, hist = numpy_newton(P)

qacc = d.qacc.numpy()[0].astype(np.float64)
c_mj = cost_grad(qacc, P)[0]
print(f"cost: NumPy {hist[-1]:.4f}  MJWarp {c_mj:.4f}  (relative gap {(c_mj - hist[-1]) / hist[-1]:.1e})")
err = np.abs(a_np - qacc).max()
print(f"max |qacc_numpy - d.qacc| = {err:.2e}   (relative {err / np.abs(qacc).max():.1e})")
force = np.where(P["J"] @ a_np - P["aref"] < 0, -P["D"] * (P["J"] @ a_np - P["aref"]), 0.0)
print(f"max |force_numpy - d.efc.force| = {np.abs(force - d.efc.force.numpy()[0, :n]).max():.2e}"
      f"  (max force {force.max():.1f} N)")

## 5. CG vs Newton
Iteration counts per world for the four (solver, cone) pairs. solver and cone are compile-time choices: each pair compiles its kernels once (slow on CPU).

In [ ]:
# Example 20.1 - ex1_niter.py
"""Example 20.1 - solver iterations per world: Newton vs CG, pyramidal vs elliptic cones.

box_stack(5) (nv 30) with njmax=128, so the 80 pyramidal rows fit. Every world starts with its own
random box velocities, so every world needs its own number of solver iterations. We step
NSTEP times for each of the four (solver, cone) combinations and print mean and max solver_niter.
solver and cone are compile-time settings: each combination compiles its own kernels once.
Run: python ex1_niter.py [--dump-json PATH]
  --dump-json PATH writes the per-config, per-world mean niter for the page's bar chart (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
rng = np.random.default_rng(0)
qvel0 = rng.normal(0.0, 0.5, size=(NWORLD, mjm.nv)).astype(np.float32)   # world-specific kicks
qvel0[0] = 0.0                                                         # world 0 starts at rest

results = {}
print(f"box_stack(5): nv={mjm.nv}, nworld={NWORLD}, {NSTEP} steps, njmax=128")
print(f"{'solver':7s} {'cone':10s} {'mean niter':>10s} {'max':>4s}  per-world mean niter")
for solver in ("NEWTON", "CG"):
    for cone in ("PYRAMIDAL", "ELLIPTIC"):
        mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
        mjm.opt.cone = getattr(mujoco.mjtCone, f"mjCONE_{cone}")
        m = mjw.put_model(mjm)
        d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
        d.qvel.assign(qvel0)
        niter = np.zeros((NSTEP, NWORLD), dtype=np.int32)
        for i in range(NSTEP):
            mjw.step(m, d)
            niter[i] = d.solver_niter.numpy()   # one small readback per step (demo only)
        per_world = niter.mean(axis=0)
        with np.printoptions(precision=2, suppress=True):   # local: no global print state
            print(f"{solver:7s} {cone.lower():10s} {niter.mean():10.2f} {niter.max():4d}  "
                  f"{per_world[:8]}")
        ov = d.overflow.numpy()
        if ov.any():
            print("  overflow:", [b.name for b in mjw.OverflowType if (ov & b).any()])
        results[f"{solver.lower()}-{cone.lower()}"] = {
            "mean": round(float(niter.mean()), 3), "max": int(niter.max()),
            "per_world": [round(float(v), 3) for v in per_world[:64]]}
print("nefc in world 0 (last step):", int(d.nefc.numpy()[0]), "rows")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "nstep": NSTEP, "configs": results}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Newton's Cholesky paths
nv <= 48: one tile; 49..96: blocked 16x16 tiles; > 96: sparse tiles. Timing is printed on a GPU only.

In [ ]:
# Example 20.11 - ex11_cholesky_path.py
"""Example 20.11 - which Newton Cholesky path a model takes, by nv.

solver.py::_cholesky_factorize_solve picks one of three paths from nv alone:
  nv <= types.H_ONE_TILE_MAX (48)            one tile: wp.tile_cholesky on the whole nv_pad^2 H
  nv <= solver._SPARSE_BLOCK_CHOLESKY_DIM (96) blocked Cholesky over 16x16 tiles
  nv >  96                                    blocked Cholesky that skips all-zero 16x16 tiles
We print the path, nv_pad, the Jacobian layout and the per-world H workspace for four models,
run one Newton forward on each (all paths work on CPU), and, on a GPU only, time the step.
For box_stack(20) we also print which 16x16 tiles of H = M + J' D J are nonzero.
Run: python ex11_cholesky_path.py [--dump-json PATH]
"""
import argparse
import json
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import solver, types  # private API: thresholds and ConstraintState

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
</mujoco>
"""


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def path_name(nv):
    if nv <= types.H_ONE_TILE_MAX:
        return "one tile"
    if nv <= solver._SPARSE_BLOCK_CHOLESKY_DIM:
        return "blocked 16x16"
    return "sparse tiles"


def dense_J(d, mjm):
    """Decode efc.J of world 0 into a dense (nefc, nv) array (dense or CSR storage)."""
    n = min(int(d.nefc.numpy()[0]), d.njmax)
    J = d.efc.J.numpy()[0]
    if J.shape[0] != 1 or d.efc.J_rownnz.shape[1] == 0:
        return J[:n, :mjm.nv].astype(np.float64)
    out = np.zeros((n, mjm.nv))
    nnz, adr = d.efc.J_rownnz.numpy()[0], d.efc.J_rowadr.numpy()[0]
    col = d.efc.J_colind.numpy()[0, 0]
    for r in range(n):
        sl = slice(adr[r], adr[r] + nnz[r])
        out[r, col[sl]] = J[0, sl]
    return out


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()
ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 1

print(f"thresholds: H_ONE_TILE_MAX = {types.H_ONE_TILE_MAX}, "
      f"_SPARSE_BLOCK_CHOLESKY_DIM = {solver._SPARSE_BLOCK_CHOLESKY_DIM}, "
      f"tile = {types.TILE_SIZE_JTDAJ_DENSE}")
print(f"{'model':14s} {'nv':>3s} {'nv_pad':>6s} {'J':>6s} {'path':>14s} {'H/world':>9s} "
      f"{'nefc':>4s} {'niter':>5s}  timing")
models = [("playground", PLAYGROUND, 8, 32), ("box_stack(5)", box_stack(5), 32, 128),
          ("box_stack(10)", box_stack(10), 64, 256), ("box_stack(20)", box_stack(20), 128, 512)]
dump = {}
for name, xml, nconmax, njmax in models:
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)                 # Newton is the default solver
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
    for _ in range(60):
        mjw.step(m, d)
    timing = "skipped on CPU: timing is GPU-only here"
    if ON_GPU:
        wp.synchronize()
        t0 = time.perf_counter()
        for _ in range(100):
            mjw.step(m, d)
        wp.synchronize()
        timing = f"{(time.perf_counter() - t0) / 100 * 1e3:.2f} ms/step, nworld={NWORLD}"
    hbytes = m.nv_pad * m.nv_pad * 4
    print(f"{name:14s} {m.nv:3d} {m.nv_pad:6d} {'sparse' if m.is_sparse else 'dense':>6s} "
          f"{path_name(m.nv):>14s} {hbytes / 1024:7.1f}KB {int(d.nefc.numpy()[0]):4d} "
          f"{int(d.solver_niter.numpy()[0]):5d}  {timing}")
    if name == "box_stack(20)":
        mjd = mujoco.MjData(mjm)
        mjw.get_data_into(mjd, mjm, d)
        M = np.zeros((mjm.nv, mjm.nv))
        mujoco.mj_fullM(mjm, mjd, M)
        J = dense_J(d, mjm)
        n = J.shape[0]
        act = d.efc.state.numpy()[0, :n] == int(types.ConstraintState.QUADRATIC)
        Dact = d.efc.D.numpy()[0, :n] * act
        H = M + J.T @ (Dact[:, None] * J)
        t = types.TILE_SIZE_JTDAJ_DENSE
        nt = m.nv_pad // t
        Hp = np.zeros((m.nv_pad, m.nv_pad))
        Hp[:mjm.nv, :mjm.nv] = H
        tiles = [[int(np.any(Hp[i*t:(i+1)*t, j*t:(j+1)*t] != 0)) for j in range(nt)]
                 for i in range(nt)]
        upper = [tiles[i][j] for i in range(nt) for j in range(i, nt)]
        print(f"  H tiles ({nt}x{nt}, upper triangle): {sum(upper)} of {len(upper)} nonzero")
        for row in tiles:
            print("   ", " ".join("#" if v else "." for v in row))
        dump = {"nv": mjm.nv, "nv_pad": m.nv_pad, "tile": t, "tiles": tiles}

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 7. State flips and the incremental Hessian
Rerun one cold-started Newton solve with iterations = 0, 1, 2, ... and count rows that changed ConstraintState. Pyramidal Newton turns each QUADRATIC flip into one rank-1 update of H.

In [ ]:
# Example 20.12 - ex12_state_flips.py
"""Example 20.12 - constraint state flips per Newton iteration (feeds the state-flip strip).

box_stack(5), one kicked mid-motion state, Newton, cold start (warmstart off). MJWarp does not
expose its per-iteration state, so we rerun the SAME solve with opt.iterations = 0, 1, 2, ... and
read d.efc.state after each run (iterations = 0 shows the states init_context computed at the
start point). A row whose state changes between iteration k-1 and k "flipped". For the pyramidal
cone, a flip into or out of QUADRATIC is exactly what makes _update_gradient_h_incremental add or
subtract one rank-1 term D_i J_i' J_i to the Hessian; iterations without flips take the stable
fast path. We repeat it for the elliptic cone, where contact rows can also be in the CONE state.
DeterminismType.CONTACTS | CONSTRAINT keeps the row order identical between the reruns (Ch 22).
Run: python ex12_state_flips.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

KMAX = 8
NAMES = {int(v): k.replace("mjCNSTRSTATE_", "")
         for k, v in mujoco.mjtConstraintState.__members__.items()}
SHORT = {"SATISFIED": "S", "QUADRATIC": "Q", "LINEARNEG": "-", "LINEARPOS": "+", "CONE": "C"}

dump = {"states": NAMES, "runs": {}}
for cone in ("PYRAMIDAL", "ELLIPTIC"):
    mjm = mujoco.MjModel.from_xml_string(box_stack(5))
    mjm.opt.cone = getattr(mujoco.mjtCone, f"mjCONE_{cone}")
    m = mjw.put_model(mjm)                       # Newton is the default solver
    m.opt.deterministic = int(mjw.DeterminismType.CONTACTS | mjw.DeterminismType.CONSTRAINT)
    d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
    d.qvel.assign(np.random.default_rng(1).normal(0.0, 0.1, size=(1, mjm.nv)).astype(np.float32))
    for _ in range(20):
        mjw.step(m, d)
    state0 = {k: getattr(d, k).numpy().copy() for k in ("qpos", "qvel", "qacc_warmstart")}

    m.opt.disableflags |= mjw.DisableBit.WARMSTART
    m.opt.warn_overflow = 0
    m.opt.tolerance = wp.array([1e-12], dtype=float)   # do not stop early while logging
    rows, niters = [], []
    for k in range(KMAX + 1):
        m.opt.iterations = k
        for key, v in state0.items():
            getattr(d, key).assign(v)
        mjw.forward(m, d)
        n = int(d.nefc.numpy()[0])
        rows.append(d.efc.state.numpy()[0, :n].copy())
        niters.append(int(d.solver_niter.numpy()[0]))
    n = len(rows[0])
    print(f"\n{cone.lower()}: nefc={n}, Newton cold start, rows by state per iteration k")
    print("  k  niter  flips  quad flips  counts")
    flips_log = []
    for k, st in enumerate(rows):
        counts = {NAMES[s]: int(c) for s, c in zip(*np.unique(st, return_counts=True))}
        if k == 0:
            flips = quad_flips = 0
            flipped = []
        else:
            prev = rows[k - 1]
            flipped = np.nonzero(st != prev)[0].tolist()
            quad = int(mujoco.mjtConstraintState.mjCNSTRSTATE_QUADRATIC)
            quad_flips = int(np.sum((st == quad) != (prev == quad)))
            flips = len(flipped)
        flips_log.append(flipped)
        cstr = ", ".join(f"{name.lower()} {c}" for name, c in counts.items())
        print(f"  {k}  {niters[k]:5d}  {flips:5d}  {quad_flips:10d}  {cstr}")
    first = "".join(SHORT[NAMES[s]] for s in rows[0][:40])
    last = "".join(SHORT[NAMES[s]] for s in rows[-1][:40])
    print(f"  first 40 rows at k=0: {first}")
    print(f"  first 40 rows at k={KMAX}: {last}")
    ov = int(d.overflow.numpy()[0])             # sticky: set by the capped runs k < niter
    print("  overflow:", [b.name for b in mjw.OverflowType if ov & b] or "none",
          "(the bits are sticky: the runs that stopped at k < 4 set ITERATIONS)")
    dump["runs"][cone.lower()] = {"nefc": n, "niter": niters,
                                  "rows": [r.astype(int).tolist() for r in rows],
                                  "flipped": flips_log}

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. The exact linesearch
The ray cost along MJWarp's first CG direction, its kinks, the exact minimizer, a NumPy replay of `_linesearch_iterative_kernel` (Newton probe from 0, then three probes per round) and the alpha MJWarp took.

In [ ]:
# Example 20.6 - ex6_linesearch_plot.py
"""Example 20.6 - the exact linesearch along MJWarp's first CG search direction.

Cold-started CG on box_stack(5) two steps after a small random kick (12 contact rows while the
bottom box meets the floor). Its first search direction is
search = -M^-1 grad(qacc_smooth). We evaluate the ray cost phi(alpha) = f(qacc_smooth + alpha *
search) in NumPy: a convex piecewise quadratic with a kink wherever a row J_i a - aref_i changes
sign. We find its exact minimizer by walking the kinks, replay MJWarp's bracketing algorithm
(_linesearch_iterative_kernel: a Newton probe from 0, then rounds of three probes lo / hi / mid)
in NumPy, and compare both with the alpha that MJWarp took (one CG iteration, iterations=1).
Run: python ex6_linesearch_plot.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
d.qvel.assign(np.random.default_rng(1).normal(0.0, 0.1, size=(1, mjm.nv)).astype(np.float32))
for _ in range(2):
    mjw.step(m, d)

mjm.opt.solver = mujoco.mjtSolver.mjSOL_CG
m = mjw.put_model(mjm)
m.opt.disableflags |= mjw.DisableBit.WARMSTART
m.opt.warn_overflow = 0
m.opt.iterations = 1                          # exactly one linesearch
mjw.forward(m, d)

mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d)
M = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, M)
n = int(d.nefc.numpy()[0])
J = d.efc.J.numpy()[0, :n, :mjm.nv].astype(np.float64)
D = d.efc.D.numpy()[0, :n].astype(np.float64)
aref = d.efc.aref.numpy()[0, :n].astype(np.float64)
a0 = d.qacc_smooth.numpy()[0].astype(np.float64)
a1 = d.qacc.numpy()[0].astype(np.float64)

jaref0 = J @ a0 - aref
grad0 = J.T @ (D * (jaref0 < 0) * jaref0)     # Gauss term is 0 at qacc_smooth
s = -np.linalg.solve(M, grad0)                # CG's first direction: -M^-1 grad
jv = J @ s
alpha_mjw = (a1 - a0) @ s / (s @ s)


def phi(t):
    """Ray cost relative to t=0, its slope and curvature (what the kernel's probes return)."""
    x = jaref0 + t * jv
    act, act0 = x < 0, jaref0 < 0
    c = 0.5 * t * t * (s @ M @ s) + 0.5 * np.sum(D[act] * x[act] ** 2) \
        - 0.5 * np.sum(D[act0] * jaref0[act0] ** 2)
    g = t * (s @ M @ s) + np.sum(D[act] * jv[act] * x[act])
    h = s @ M @ s + np.sum(D[act] * jv[act] ** 2)
    return np.array([c, g, h])


# exact minimizer: the slope is piecewise linear and increasing; find its zero
with np.errstate(divide="ignore", invalid="ignore"):
    kinks = np.where(jv != 0, -jaref0 / jv, np.inf)
kinks = np.sort(kinks[(kinks > 0) & np.isfinite(kinks)])
edges = np.concatenate([[0.0], kinks, [kinks[-1] * 2 + 1.0]])
t_exact = None
for lo, hi in zip(edges[:-1], edges[1:]):
    mid = phi(0.5 * (lo + hi))
    t = 0.5 * (lo + hi) - mid[1] / mid[2]
    if lo <= t <= hi:
        t_exact = t
        break

# NumPy replay of the kernel's bracketing (gtol as in the kernel; noise-floor terms omitted)
tol, ls_tol = 1e-6, 0.01
scale = 1.0 / np.trace(M)
gtol = tol * ls_tol * np.linalg.norm(s) / scale
p0 = phi(0.0)
p0_delta = np.array([0.0, p0[1], p0[2]])
frames = [{"what": "p0", "alpha": 0.0, "pt": p0.tolist()}]
lo_alpha_in = -p0[1] / p0[2]
lo_in = phi(lo_alpha_in)
frames.append({"what": "newton_from_0", "alpha": lo_alpha_in, "pt": lo_in.tolist()})


def in_bracket(x, y):
    return (x[1] < y[1] < 0) or (x[1] > y[1] > 0)


alpha, rounds = 0.0, 0
if abs(lo_in[1]) < gtol and lo_in[0] < 0:
    alpha = lo_alpha_in
else:
    lo_less = lo_in[1] < p0[1]
    lo, lo_a = (lo_in, lo_alpha_in) if lo_less else (p0_delta, 0.0)
    hi, hi_a = (p0_delta, 0.0) if lo_less else (lo_in, lo_alpha_in)
    for rounds in range(1, 51):               # m.opt.ls_iterations = 50
        a_lo, a_hi, a_mid = lo_a - lo[1] / lo[2], hi_a - hi[1] / hi[2], 0.5 * (lo_a + hi_a)
        p_lo, p_hi, p_mid = phi(a_lo), phi(a_hi), phi(a_mid)
        frames.append({"what": "round", "round": rounds, "lo": lo_a, "hi": hi_a,
                       "probes": [[a_lo] + p_lo.tolist(), [a_hi] + p_hi.tolist(),
                                  [a_mid] + p_mid.tolist()]})
        conv = [(p[0], a) for p, a in ((p_lo, a_lo), (p_hi, a_hi), (p_mid, a_mid))
                if abs(p[1]) < gtol and p[0] < 0]
        if conv:
            alpha = min(conv)[1]
            break
        sw_lo = sw_hi = False
        for p, a in ((p_lo, a_lo), (p_mid, a_mid), (p_hi, a_hi)):
            if in_bracket(lo, p):
                lo, lo_a, sw_lo = p, a, True
        for k, (p, a) in enumerate(((p_hi, a_hi), (p_mid, a_mid), (p_lo, a_lo))):
            if in_bracket(hi, p) or (k == 0 and hi[1] < 0 < p[1]):
                hi, hi_a, sw_hi = p, a, True
        alpha = lo_a if lo[0] < hi[0] else hi_a
        if not sw_lo and not sw_hi:
            break

print(f"nefc={n}, kinks on the ray at alpha > 0: {len(kinks)} (first {kinks[0]:.4f}, "
      f"last {kinks[-1]:.4f})")
print(f"slope at 0: {p0[1]:.4e}  (negative: search is a descent direction)")
print(f"Newton probe from 0 (one quadratic piece): alpha = {lo_alpha_in:.5f}, "
      f"slope there {lo_in[1]:+.3e}, |slope| < gtol={gtol:.3e}? {abs(lo_in[1]) < gtol}")
print(f"bracketing rounds used: {rounds}  -> alpha = {alpha:.5f}")
print(f"exact minimizer (kink walk):       alpha = {t_exact:.5f}")
print(f"MJWarp's step (qacc1 - qacc0)/search: alpha = {alpha_mjw:.5f}")
print(f"cost drop phi(alpha_mjw) = {phi(alpha_mjw)[0]:.2f}  vs exact {phi(t_exact)[0]:.2f}")

if args.dump_json:
    ts = np.linspace(0.0, 1.6 * t_exact, 161)
    curve = [[round(float(t), 5), round(float(phi(t)[0]), 3)] for t in ts]
    with open(args.dump_json, "w") as f:
        json.dump({"curve": curve, "kinks": [round(float(k), 5) for k in kinks if k < ts[-1]],
                   "frames": json.loads(json.dumps(frames, default=float)),
                   "alpha_exact": round(float(t_exact), 6), "alpha_mjw": round(float(alpha_mjw), 6),
                   "gtol": float(gtol)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Ray cost from Example 20.6 (run it first: it defines phi, kinks, t_exact, alpha_mjw).
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

ts = np.linspace(0.0, 1.6 * t_exact, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(ts, [phi(t)[0] for t in ts], label="phi(alpha)")
for k in kinks[kinks < ts[-1]]:
    plt.axvline(k, color="orange", lw=.8, ls=":")
plt.axvline(alpha_mjw, color="green", label="MJWarp alpha")
plt.xlabel("alpha"); plt.ylabel("cost change"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 9. Termination
Log one cold-started solve per iteration and check which `_solve_done` test fires (alpha == 0, improvement, gradient, Newton's model improvement), then the per-world niter of 64 worlds.

In [ ]:
# Example 20.4 - ex4_convergence_log.py
"""Example 20.4 - convergence log: scaled gradient norm and improvement per iteration, CG vs Newton.

MJWarp does not expose its solver context, so we rerun ONE solve from the same state with
opt.iterations = 0, 1, 2, ... and rebuild the solver's own quantities in NumPy after each run:
  cost(a)  = 0.5 (a - qacc_smooth)' M (a - qacc_smooth) + sum_active 0.5 D (J a - aref)^2
  grad     = M a - qfrc_smooth - J' force               (scaled by inertia_scale = 1 / trace(M))
  decrement= grad' H^-1 grad with H = M + J' diag(D active) J    (Newton's model improvement)
The solve is cold-started (warmstart off) so that every iteration is visible. Then we mark which
_solve_done test would stop the solver at the default tolerance 1e-6, and finally collect the
per-world solver_niter of a 64-world batch (after 10 ordinary warm-started steps) for the histogram.
Run: python ex4_convergence_log.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

TOL = 1e-6
NWORLD_GRID = 64
mjm = mujoco.MjModel.from_xml_string(box_stack(5))
mjd = mujoco.MjData(mjm)

# one kicked, mid-motion state shared by both solvers
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
d.qvel.assign(np.random.default_rng(1).normal(0.0, 0.1, size=(1, mjm.nv)).astype(np.float32))
for _ in range(20):
    mjw.step(m, d)
state = {k: getattr(d, k).numpy().copy() for k in ("qpos", "qvel", "qacc_warmstart")}


def numpy_metrics(d):
    """Rebuild cost, scaled gradient norm and Newton decrement for world 0."""
    mjw.get_data_into(mjd, mjm, d)
    M = np.zeros((mjm.nv, mjm.nv))
    mujoco.mj_fullM(mjm, mjd, M)
    n = int(d.nefc.numpy()[0])
    J = d.efc.J.numpy()[0, :n, :mjm.nv].astype(np.float64)
    D = d.efc.D.numpy()[0, :n].astype(np.float64)
    aref = d.efc.aref.numpy()[0, :n].astype(np.float64)
    a = d.qacc.numpy()[0].astype(np.float64)
    a0 = d.qacc_smooth.numpy()[0].astype(np.float64)
    jaref = J @ a - aref
    active = jaref < 0.0                     # pyramidal rows: QUADRATIC when J a - aref < 0
    cost = 0.5 * (a - a0) @ M @ (a - a0) + 0.5 * np.sum(D[active] * jaref[active] ** 2)
    force = np.where(active, -D * jaref, 0.0)
    grad = M @ (a - a0) - J.T @ force        # = M a - qfrc_smooth - qfrc_constraint
    H = M + J.T @ (D[:, None] * active[:, None] * J)
    scale = 1.0 / np.trace(M)
    return cost, scale, scale * np.linalg.norm(grad), scale * 0.5 * grad @ np.linalg.solve(H, grad)


log = {}
for solver, kmax in (("CG", 50), ("NEWTON", 6)):
    mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    m.opt.disableflags |= mjw.DisableBit.WARMSTART
    m.opt.tolerance = wp.array([1e-12], dtype=float)   # never stop early while logging
    rows, prev_cost = [], None
    for k in range(kmax + 1):
        m.opt.iterations = k
        for key, v in state.items():
            getattr(d, key).assign(v)
        mjw.forward(m, d)
        cost, scale, gnorm, dec = numpy_metrics(d)
        impr = 0.0 if prev_cost is None else scale * (prev_cost - cost)
        prev_cost = cost
        rows.append({"k": k, "cost": cost, "grad": gnorm, "improvement": impr, "decrement": dec})

    # which _solve_done test fires first at the default tolerance?
    fired = None
    for r in rows[1:]:
        tests = [("improvement", 0.0 < r["improvement"] < TOL), ("gradient", r["grad"] < TOL)]
        if solver == "NEWTON":
            tests.append(("model improvement (0.5 decrement)", r["decrement"] < TOL))
        hit = [name for name, ok in tests if ok]
        if hit:
            fired = (r["k"], hit)
            break
    m.opt.tolerance = wp.array([TOL], dtype=float)
    m.opt.iterations = 100
    for key, v in state.items():
        getattr(d, key).assign(v)
    mjw.forward(m, d)
    actual = int(d.solver_niter.numpy()[0])

    print(f"\n{solver}: cold start, nefc={int(d.nefc.numpy()[0])}, scale = 1/trace(M) = {scale:.4f}")
    print("  k        cost    log10|grad|s   log10 improvement")
    for r in rows:
        if solver == "CG" and r["k"] > 12 and r["k"] % 4:
            continue                          # print a subset of the long CG log
        li = f"{np.log10(r['improvement']):8.2f}" if r["improvement"] > 0 else "       -"
        print(f"  {r['k']:2d} {r['cost']:12.4f} {np.log10(r['grad']):12.2f}   {li}")
    print(f"  first test to pass at tol={TOL:g}: iteration {fired[0]} -> {', '.join(fired[1])}")
    print(f"  MJWarp solver_niter at tol={TOL:g}: {actual}")
    log[solver.lower()] = {"rows": [{kk: (float(f"{v:.6g}") if kk != "k" else v)
                                     for kk, v in r.items()} for r in rows],
                           "fired": fired[0], "test": fired[1][0], "niter": actual}

# per-world histogram: 64 worlds with their own kicks, after 10 ordinary (warm-started) steps
hist = {}
for solver in ("NEWTON", "CG"):
    mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
    m = mjw.put_model(mjm)
    dg = mjw.make_data(mjm, nworld=NWORLD_GRID, nconmax=32, njmax=128)
    kicks = np.random.default_rng(2).normal(0.0, 0.5, size=(NWORLD_GRID, mjm.nv))
    dg.qvel.assign(kicks.astype(np.float32))
    for _ in range(10):
        mjw.step(m, dg)
    niter = dg.solver_niter.numpy()
    vals, counts = np.unique(niter, return_counts=True)
    print(f"\n{solver} per-world solver_niter, {NWORLD_GRID} worlds, step 10:")
    print("  niter:", dict(zip(vals.tolist(), counts.tolist())), " max:", int(niter.max()),
          " overflow:", "none" if not dg.overflow.numpy().any() else dg.overflow.numpy())
    hist[solver.lower()] = niter.tolist()

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"tol": TOL, "log": log, "grid": hist}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Convergence plot from Example 20.4 (run it first: it defines `log`, `hist` and TOL).
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, color in (("cg", "C0"), ("newton", "C1")):
    rows = log[name]["rows"]
    k = [r["k"] for r in rows]
    ax[0].semilogy(k, [r["grad"] for r in rows], "o-", color=color, ms=3, label=name)
    imp = [(r["k"], r["improvement"]) for r in rows if r["improvement"] > 0]
    ax[0].semilogy([a for a, _ in imp], [b for _, b in imp], "--", color=color, alpha=.6)
ax[0].axhline(TOL, color="red", ls=":", label="tolerance")
ax[0].set_xlabel("iteration k"); ax[0].set_ylabel("scaled gradient (solid), improvement (dashed)")
ax[0].legend(); ax[0].grid(alpha=.3)
for name, color in (("newton", "C1"), ("cg", "C0")):
    ax[1].hist(hist[name], bins=np.arange(0, max(hist[name]) + 2) - 0.5, color=color, alpha=.7, label=name)
ax[1].set_xlabel("solver_niter"); ax[1].set_ylabel("worlds"); ax[1].legend()
plt.tight_layout(); plt.show()

`m.opt.tolerance` is per world (`tolerance[worldid % len]`). put_model clamps the MJCF value to >= 1e-6; assigning an array afterwards bypasses the clamp.

In [ ]:
# Example 20.2 - ex2_tolerance_per_world.py
"""Example 20.2 - a different solver tolerance in every world.

m.opt.tolerance is a batched option (shape ("*",)): the solver kernels read
tolerance[worldid % tolerance.shape[0]]. All four worlds hold the same falling box stack, so the
only difference is the tolerance. We print solver_niter for one cold-started solve (warmstart off,
so the solver starts from qacc_smooth) and for a warm-started one.
Note: put_model clamps the MJCF tolerance to >= 1e-6 (float32); assigning the array afterwards
bypasses that clamp, which is how 1e-8 gets in here.
Run: python ex2_tolerance_per_world.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


TOLS = [1e-2, 1e-4, 1e-6, 1e-8]
mjm = mujoco.MjModel.from_xml_string(box_stack(5))
print("MJCF tolerance:", mjm.opt.tolerance)

for solver in ("NEWTON", "CG"):
    mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    if solver == "NEWTON":
        print("after put_model:", m.opt.tolerance.numpy(), "(clamped to >= 1e-6)")
    d = mjw.make_data(mjm, nworld=len(TOLS), nconmax=32, njmax=128)
    for _ in range(20):                       # let the stack land: 80 active rows per world
        mjw.step(m, d)
    m.opt.tolerance = wp.array(TOLS, dtype=float)   # one value per world

    m.opt.disableflags |= mjw.DisableBit.WARMSTART
    mjw.forward(m, d)
    cold = d.solver_niter.numpy().copy()
    m.opt.disableflags &= ~mjw.DisableBit.WARMSTART
    mjw.forward(m, d)
    warm = d.solver_niter.numpy().copy()
    print(f"{solver:6s} tolerance {TOLS}")
    print(f"       niter cold start (qacc_smooth): {cold}")
    print(f"       niter warm start (last qacc):   {warm}")
    print("       nefc:", d.nefc.numpy(), " ITERATIONS bit:",
          bool((d.overflow.numpy() & mjw.OverflowType.ITERATIONS).any()))

A capped CG returns a wrong qacc and sets ITERATIONS; `ls_iterations` is a compile-time constant (this cell recompiles the linesearch) and LS_ITERATIONS is bit 1 << 10. The kernel printf appears in the cell output.

In [ ]:
# Example 20.3 - ex3_iteration_cap.py
"""Example 20.3 - capping the solver: opt.iterations, ITERATIONS and LS_ITERATIONS overflow bits.

CG on a landed box_stack(5), cold-started (warmstart off) so it needs many iterations.
Part 1 caps opt.iterations at 1, 5, 20 and 100 (a runtime value: no recompile) and prints
solver_niter, the ITERATIONS bit and the z acceleration of the bottom box against a converged
Newton reference. Part 2 sets opt.ls_iterations = 1 (a compile-time constant of the linesearch
kernel: this recompiles it) and shows the LS_ITERATIONS bit (1 << 10) and the kernel's printf.
Run: python ex3_iteration_cap.py
"""
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def bits(d):
    ov = int(d.overflow.numpy()[0])
    return [b.name for b in mjw.OverflowType if ov & b] or ["none"]


mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)                       # Newton (default) for the settle run and reference
d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
for _ in range(20):
    mjw.step(m, d)
m.opt.disableflags |= mjw.DisableBit.WARMSTART
mjw.forward(m, d)
ref = d.qacc.numpy()[0, 2]
print(f"Newton reference: niter {d.solver_niter.numpy()[0]}, qacc[2] = {ref:.5f}")

mjm.opt.solver = mujoco.mjtSolver.mjSOL_CG
m = mjw.put_model(mjm)
m.opt.disableflags |= mjw.DisableBit.WARMSTART
m.opt.warn_overflow = 0                      # silence printf for part 1 (read the bits instead)
print("\nPart 1: CG with an iteration cap")
for cap in (1, 5, 20, 100):
    m.opt.iterations = cap
    d2 = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)   # fresh Data: bits are sticky
    for name in ("qpos", "qvel", "qacc_warmstart"):
        getattr(d2, name).assign(getattr(d, name).numpy())
    mjw.forward(m, d2)
    q2 = d2.qacc.numpy()[0, 2]
    print(f"  iterations={cap:3d}: niter {d2.solver_niter.numpy()[0]:3d}  overflow {bits(d2)}  "
          f"qacc[2] = {q2:9.5f}  (error {abs(q2 - ref):.1e})")

print("\nPart 2: ls_iterations = 1 (recompiles the linesearch kernel)")
m.opt.iterations = 100
m.opt.disableflags &= ~mjw.DisableBit.WARMSTART
m.opt.ls_iterations = 1
d3 = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
rng = np.random.default_rng(0)
d3.qvel.assign(rng.normal(0.0, 0.5, size=(1, mjm.nv)).astype(np.float32))   # a kicked stack
for step in range(40):
    snap = {k: getattr(d3, k).numpy().copy() for k in ("qpos", "qvel", "qacc_warmstart")}
    mjw.step(m, d3)
    if d3.overflow.numpy()[0] & mjw.OverflowType.LS_ITERATIONS:
        break
print(f"  first LS_ITERATIONS overflow at step {step}: overflow {bits(d3)}")

# replay that step with the warning switched on (warn_overflow is read through wp.static:
# changing it recompiles the kernels that print)
m.opt.warn_overflow = int(mjw.OverflowType.LS_ITERATIONS)
d4 = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
for k, v in snap.items():
    getattr(d4, k).assign(v)
sys.stdout.flush()
mjw.forward(m, d4)
wp.synchronize()                             # the kernel printf goes straight to the C stdout
sys.stdout.flush()
ov = int(d4.overflow.numpy()[0])
print(f"  replayed: niter {d4.solver_niter.numpy()[0]}, overflow = {ov} -> {bits(d4)}")
print("  LS_ITERATIONS set:", bool(ov & mjw.OverflowType.LS_ITERATIONS), f"(1 << 10 = {1 << 10})")

## 10. Knobs and their cost
Eager step time for iterations {1, 5, 20, 50} x ls_iterations {5, 20}. On a GPU, run this and read your own numbers.

In [ ]:
# Example 20.10 - ex10_solver_timing.py
"""Example 20.10 - eager step time vs opt.iterations and opt.ls_iterations (CG and Newton).

opt.iterations is a runtime value; opt.ls_iterations is a compile-time constant of the
linesearch kernel, so each new value compiles once (we warm up every configuration first).
Eager stepping (no CUDA graph): the solver loop is wp.capture_while, which outside a graph
capture is a host loop that reads nsolving after every iteration and stops when all worlds are
done. On a CPU the numbers are CPU numbers and say nothing about a GPU; on Colab run it on a GPU.
Run: python ex10_solver_timing.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200 if ON_GPU else 20

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
qvel0 = np.random.default_rng(0).normal(0.0, 0.5, size=(NWORLD, mjm.nv)).astype(np.float32)

print(f"device {wp.get_device()}, box_stack(5), nworld={NWORLD}, {NSTEP} timed steps per row")
print(f"{'solver':7s} {'iters':>5s} {'ls_iters':>8s} {'ms/step':>8s} {'world-steps/s':>14s} "
      f"{'niter (last step)':>17s} {'ITER bit':>8s}")
for solver in ("CG", "NEWTON"):
    mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    for ls_iters in (5, 20):
        m.opt.ls_iterations = ls_iters
        for iters in (1, 5, 20, 50):
            m.opt.iterations = iters
            d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
            d.qvel.assign(qvel0)
            for _ in range(3):                 # warm-up: compiles the new ls_iterations once
                mjw.step(m, d)
            wp.synchronize()
            t0 = time.perf_counter()
            for _ in range(NSTEP):
                mjw.step(m, d)
            wp.synchronize()
            dt = (time.perf_counter() - t0) / NSTEP
            it_bit = bool((d.overflow.numpy() & mjw.OverflowType.ITERATIONS).any())
            print(f"{solver:7s} {iters:5d} {ls_iters:8d} {dt * 1e3:8.2f} {NWORLD / dt:14,.0f} "
                  f"{d.solver_niter.numpy().mean():17.2f} {str(it_bit):>8s}")

## 11. Per-world convergence
`capture_while(nsolving, ...)` runs until the slowest world is done (see the histogram above). With `graph_conditional=False` a fixed loop of `opt.iterations` runs. Graph capture itself is Chapter 24.

## 12. Warmstart and run_qacc_smooth
`qacc_warmstart` is written by `step` only. 4.0.1 never picks the better of warmstart and qacc_smooth.

In [ ]:
# Example 20.7 - ex7_warmstart.py
"""Example 20.7 - what warmstart buys: solver_niter with and without DisableBit.WARMSTART.

With warmstart on (the default), _solve_init_dof starts each world from qacc_warmstart, the qacc
of the previous step (written by step after integration), whenever the world has constraint rows.
With it off, every solve starts from qacc_smooth. Two scenes: box_stack(5) worlds that get random
kicks (lots of motion) and the same stack resting from the start (the common case in RL: most
contacts persist from step to step).
Run: python ex7_warmstart.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
kicked = np.random.default_rng(0).normal(0.0, 0.5, size=(NWORLD, mjm.nv)).astype(np.float32)
resting = np.zeros_like(kicked)

print(f"box_stack(5), nworld={NWORLD}, {NSTEP} steps; niter over all worlds and steps")
for scene, qvel0, solver in (("kicked", kicked, "NEWTON"), ("kicked", kicked, "CG"),
                             ("resting", resting, "NEWTON"), ("resting", resting, "CG")):
    mjm.opt.solver = getattr(mujoco.mjtSolver, f"mjSOL_{solver}")
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    for warm in (True, False):
        if warm:
            m.opt.disableflags &= ~mjw.DisableBit.WARMSTART
        else:
            m.opt.disableflags |= mjw.DisableBit.WARMSTART
        d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
        d.qvel.assign(qvel0)
        niter = []
        for _ in range(NSTEP):
            mjw.step(m, d)
            niter.append(d.solver_niter.numpy().copy())
        niter = np.concatenate(niter)
        q = np.percentile(niter, [50, 90, 100]).astype(int)
        label = "warmstart on " if warm else "warmstart off"
        print(f"  {scene:7s} {solver:6s} {label}: mean {niter.mean():6.2f}  median {q[0]:3d}  "
              f"p90 {q[1]:3d}  max {q[2]:3d}  ITERATIONS bit: "
              f"{bool((d.overflow.numpy() & mjw.OverflowType.ITERATIONS).any())}")

Experimental: `run_qacc_smooth=False` with Newton skips the M factorization and the qacc_smooth solve.

In [ ]:
# Example 20.8 - ex8_run_qacc_smooth.py
"""Example 20.8 - EXPERIMENTAL: opt.run_qacc_smooth = False (warmstarted Newton without qacc_smooth).

With run_qacc_smooth False (and Newton, warmstart on, noslip off, iterations > 0, njmax > 0:
smooth.skip_qacc_smooth), fwd_acceleration skips smooth.factor_solve_i: no M factorization and
no qacc_smooth solve. Newton then starts every world from qacc_warmstart and never needs M^-1.
We run box_stack(5) for 200 steps both ways from identical kicked states and compare the time
per step (CPU here: a CPU number says nothing about a GPU) and the trajectories.
Run: python ex8_run_qacc_smooth.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import smooth  # private API: only to print the skip test


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200

mjm = mujoco.MjModel.from_xml_string(box_stack(5))      # Newton is the default solver
qvel0 = np.random.default_rng(0).normal(0.0, 0.2, size=(NWORLD, mjm.nv)).astype(np.float32)

runs = {}
for flag in (True, False):
    m = mjw.put_model(mjm)
    m.opt.run_qacc_smooth = flag
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    print(f"run_qacc_smooth={flag}: skip_qacc_smooth -> {smooth.skip_qacc_smooth(m, d)}")
    d.qvel.assign(qvel0)
    mjw.step(m, d)                                       # warm up (JIT compile)
    d.qvel.assign(qvel0)
    d.qpos.assign(np.tile(mjm.qpos0, (NWORLD, 1)))
    d.qacc_warmstart.zero_()
    d.time.zero_()
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NSTEP):
        mjw.step(m, d)                                   # no host readback inside the timed loop
    wp.synchronize()
    dt = (time.perf_counter() - t0) / NSTEP
    runs[flag] = d.qpos.numpy().copy()
    print(f"  {dt * 1e3:7.2f} ms/step (device {wp.get_device()}), mean niter "
          f"(last step) {d.solver_niter.numpy().mean():.2f}, overflow "
          f"{'none' if not d.overflow.numpy().any() else d.overflow.numpy()}")
    print(f"  max |d.qacc_smooth| = {np.abs(d.qacc_smooth.numpy()).max():.3f}"
          f"{'  (never computed: stale)' if not flag else ''}")

dq = np.abs(runs[True] - runs[False])
print(f"after {NSTEP} steps: max |dqpos| = {dq.max():.2e}, mean = {dq.mean():.2e}")
print("top box z, first 4 worlds (default):     ", np.round(runs[True][:4, 4 * 7 + 2], 4))
print("top box z, first 4 worlds (experimental):", np.round(runs[False][:4, 4 * 7 + 2], 4))

## 13. noslip
PGS over the friction rows with the unregularized Delassus operator, after the main solve. Its sweeps are added to `solver_niter`. Compared with C MuJoCo on the same model.

In [ ]:
# Example 20.9 - ex9_noslip.py
"""Example 20.9 - noslip: the slow creep of soft friction, and what noslip removes.

A flat slab (0.2 x 0.2 x 0.04 m, 1 kg, friction 0.8) rests on a 30-degree slope. The slope is
modelled by tilting gravity: g = 9.81 (sin 30, 0, -cos 30). tan 30 = 0.58 < 0.8, so a rigid model
would hold the slab still. MuJoCo's friction rows are soft (regularized): they only resist
sliding through a small velocity, so the slab creeps. noslip (solver.py::noslip) runs projected
Gauss-Seidel over the friction rows only, with the unregularized Delassus operator J M^-1 J',
after the main solve; noslip sweeps are added to d.solver_niter. We print the drift after 2 s for
both cones, with and without noslip, next to C MuJoCo (float64) for the same model.
Run: python ex9_noslip.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SLOPE = """
<mujoco model="slope">
  <option timestep="0.005" gravity="4.905 0 -8.496"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 .1" friction="0.8"/>
    <body name="slab" pos="0 0 0.02">
      <freejoint/>
      <geom type="box" size=".1 .1 .02" mass="1" friction="0.8"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
T_END = 2.0

print(f"drift down the slope in {T_END:.0f} s (after 20 settling steps)")
for cone in ("PYRAMIDAL", "ELLIPTIC"):
    for noslip in (0, 10):
        mjm = mujoco.MjModel.from_xml_string(SLOPE)
        mjm.opt.cone = getattr(mujoco.mjtCone, f"mjCONE_{cone}")
        mjm.opt.noslip_iterations = noslip
        nstep = int(round(T_END / mjm.opt.timestep))

        m = mjw.put_model(mjm)
        d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
        for _ in range(20):
            mjw.step(m, d)
        x0 = d.qpos.numpy()[0, 0]
        for _ in range(nstep):
            mjw.step(m, d)
        drift = d.qpos.numpy()[0, 0] - x0

        mjd = mujoco.MjData(mjm)                 # the same model in C MuJoCo, for reference
        for _ in range(20):
            mujoco.mj_step(mjm, mjd)
        cx0 = mjd.qpos[0]
        for _ in range(nstep):
            mujoco.mj_step(mjm, mjd)

        print(f"  {cone.lower():9s} noslip_iterations={noslip:2d}: MJWarp {drift * 1e3:7.3f} mm  "
              f"(C MuJoCo {(mjd.qpos[0] - cx0) * 1e3:7.3f} mm)  solver_niter "
              f"{d.solver_niter.numpy()[0]:2d}  nefc {d.nefc.numpy()[0]:2d}  overflow "
              f"{'none' if not d.overflow.numpy().any() else d.overflow.numpy()}")
print(f"noslip_tolerance: {m.opt.noslip_tolerance.numpy()[0]:.0e}")

## Exercise 20.1: settle a stiff grasp
Pick solver, cone, iterations and tolerance so that the mean niter is below 10, no ITERATIONS bit is set and the box is still held.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

GRASP = """
<mujoco model="stiff_grasp">
  <option timestep="0.005"/>
  <default><geom solref="0.004 1" friction="1.5"/></default>
  <worldbody>
    <geom name="floor" type="plane" size="2 2 .1"/>
    <body name="left" pos="-0.16 0 0.3">
      <joint name="lx" type="slide" axis="1 0 0"/>
      <geom type="box" size=".05 .1 .1" mass="1"/>
    </body>
    <body name="right" pos="0.16 0 0.3">
      <joint name="rx" type="slide" axis="1 0 0"/>
      <geom type="box" size=".05 .1 .1" mass="1"/>
    </body>
    <body name="obj" pos="0 0 0.3">
      <freejoint/>
      <geom type="box" size=".1 .08 .08" mass="0.5"/>
    </body>
  </worldbody>
  <actuator>
    <position joint="lx" kp="2000" ctrlrange="-1 1"/>
    <position joint="rx" kp="2000" ctrlrange="-1 1"/>
  </actuator>
</mujoco>
"""


def run(solver, cone, iterations, tolerance, nworld=4, nstep=60):
    """Squeeze a kicked box between two stiff pads; return mean niter, ITERATIONS bit, box z."""
    mjm = mujoco.MjModel.from_xml_string(GRASP)
    mjm.opt.solver, mjm.opt.cone = solver, cone
    mjm.opt.iterations, mjm.opt.tolerance = iterations, tolerance
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    d = mjw.make_data(mjm, nworld=nworld, nconmax=16, njmax=64)
    d.ctrl.assign(np.tile([0.02, -0.02], (nworld, 1)))         # pads press inwards
    qvel = np.zeros((nworld, mjm.nv), np.float32)
    qvel[:, 2:] = np.random.default_rng(0).normal(0, 1.0, (nworld, 6))   # kick the box
    d.qvel.assign(qvel)
    niter = []
    for _ in range(nstep):
        mjw.step(m, d)
        niter.append(d.solver_niter.numpy().copy())
    it_bit = bool((d.overflow.numpy() & mjw.OverflowType.ITERATIONS).any())
    return float(np.mean(niter)), it_bit, d.qpos.numpy()[:, 4]


# baseline: CG capped at 20 iterations
print("baseline:", run(mujoco.mjtSolver.mjSOL_CG, mujoco.mjtCone.mjCONE_PYRAMIDAL, 20, 1e-8))
settings = dict(solver=..., cone=..., iterations=..., tolerance=...)   # your choice
mean_niter, it_bit, z = run(**settings)
print(f"mean niter {mean_niter:.2f}, ITERATIONS bit {it_bit}, box z {np.round(z, 3)}")
assert mean_niter < 10 and not it_bit and (z > 0.25).all()
print("grasp solved")

In [ ]:
#@title Exercise 20.1: solution (double-click to show the code)
settings = dict(solver=mujoco.mjtSolver.mjSOL_NEWTON, cone=mujoco.mjtCone.mjCONE_PYRAMIDAL,
                iterations=100, tolerance=1e-8)
mean_niter, it_bit, z = run(**settings)
print(f"mean niter {mean_niter:.2f}, ITERATIONS bit {it_bit}, box z {np.round(z, 3)}")
assert mean_niter < 10 and not it_bit and (z > 0.25).all()
print("grasp solved")

## Exercise 20.2: your Newton matches MJWarp
Uses `cost_grad`, `exact_linesearch` and `box_stack` from Example 20.5 (run it first). Write the Newton loop and assert max |a - d.qacc| < 1e-4 on box_stack(3).

In [ ]:
# Uses cost_grad, exact_linesearch and box_stack from Example 20.5 (run that cell first).
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def newton(P, tol=1e-10, maxiter=30):
    a = P["a0"].copy()
    for _ in range(maxiter):
        ...   # H, search, linesearch, update a, stop test
    return a


mjm = mujoco.MjModel.from_xml_string(box_stack(3))
m = mjw.put_model(mjm)
m.opt.disableflags |= mjw.DisableBit.WARMSTART
d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
for _ in range(20):
    mjw.step(m, d)
mjw.forward(m, d)
mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d)
M = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, M)
n = int(d.nefc.numpy()[0])
P = {"M": M, "J": d.efc.J.numpy()[0, :n, :mjm.nv].astype(np.float64),
     "D": d.efc.D.numpy()[0, :n].astype(np.float64),
     "aref": d.efc.aref.numpy()[0, :n].astype(np.float64),
     "a0": d.qacc_smooth.numpy()[0].astype(np.float64)}
a = newton(P)
err = np.abs(a - d.qacc.numpy()[0]).max()
print(f"nefc {n}, max |a_numpy - d.qacc| = {err:.2e}")
assert err < 1e-4
print("NumPy Newton matches MJWarp")

In [ ]:
#@title Exercise 20.2: solution (double-click to show the code)
def newton(P, tol=1e-10, maxiter=30):
    a = P["a0"].copy()
    for _ in range(maxiter):
        c, g, act = cost_grad(a, P)
        H = P["M"] + P["J"].T @ (P["D"][:, None] * act[:, None] * P["J"])
        s = -np.linalg.solve(H, g)
        t, _ = exact_linesearch(a, s, P)
        a = a + t * s
        if np.linalg.norm(cost_grad(a, P)[1]) / np.trace(P["M"]) < tol:
            break
    return a


a = newton(P)
err = np.abs(a - d.qacc.numpy()[0]).max()
print(f"nefc {n}, max |a_numpy - d.qacc| = {err:.2e}")
assert err < 1e-4
print("NumPy Newton matches MJWarp")

## Exercise 20.3: predict the Cholesky path and the H memory
Write `cholesky_path(nv)` and `nv_pad(nv)` from the rules in section 6 and check them against put_model.

In [ ]:
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import solver, types   # private API: the thresholds


def cholesky_path(nv):
    ...   # "one tile", "blocked" or "sparse tiles"


def nv_pad(nv):
    ...   # Newton: multiple of 4 up to 48, else round_up(nv + 1, 16)


def h_bytes(nv, nworld):
    return nworld * nv_pad(nv) ** 2 * 4


assert [cholesky_path(n) for n in (13, 48, 49, 96, 97)] == \
    ["one tile", "one tile", "blocked", "blocked", "sparse tiles"]
for n_boxes in (2, 5, 10):                       # nv 12, 30, 60
    xml = "".join(f'<body pos="0 0 {0.3 * i + 0.2}"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
                  for i in range(n_boxes))
    mjm = mujoco.MjModel.from_xml_string(f"<mujoco><worldbody>{xml}</worldbody></mujoco>")
    assert mjw.put_model(mjm).nv_pad == nv_pad(mjm.nv), mjm.nv
print("humanoid-sized H (nv 27) in 8192 worlds:", h_bytes(27, 8192) / 2**20, "MiB")

In [ ]:
#@title Exercise 20.3: solution (double-click to show the code)
def cholesky_path(nv):
    if nv <= types.H_ONE_TILE_MAX:
        return "one tile"
    if nv <= solver._SPARSE_BLOCK_CHOLESKY_DIM:
        return "blocked"
    return "sparse tiles"


def nv_pad(nv):
    if nv <= types.H_ONE_TILE_MAX:
        return -(-nv // 4) * 4
    return -(-(nv + 1) // 16) * 16


def h_bytes(nv, nworld):
    return nworld * nv_pad(nv) ** 2 * 4


assert [cholesky_path(n) for n in (13, 48, 49, 96, 97)] == ["one tile", "one tile", "blocked", "blocked", "sparse tiles"]
for n_boxes in (2, 5, 10):                       # nv 12, 30, 60
    xml = "".join(f'<body pos="0 0 {0.3 * i + 0.2}"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
                  for i in range(n_boxes))
    mjm = mujoco.MjModel.from_xml_string(f"<mujoco><worldbody>{xml}</worldbody></mujoco>")
    assert mjw.put_model(mjm).nv_pad == nv_pad(mjm.nv), mjm.nv
print("humanoid-sized H (nv 27) in 8192 worlds:", h_bytes(27, 8192) / 2**20, "MiB")

## Stretch: Newton for elliptic cones in NumPy
Extend Example 20.5 to `mjCONE_ELLIPTIC`: group the rows of each contact, implement the SATISFIED / QUADRATIC / CONE zones of `_eval_constraint`, their gradient and Hessian block, and a linesearch that handles the cone piece. Compare the final cost with MJWarp's.

In [ ]:
# your elliptic NumPy Newton here